<a href="https://colab.research.google.com/github/jptrainor4-glitch/Data-Analytics/blob/main/Bar_Seats_per_sensor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#For each sensor, total the bar seats within 200m and within 300m, once with the outlier venues and once without. That gives the supply side for the gap score.

In [2]:
import pandas as pd
import os
from google.colab import files, drive
import matplotlib.pyplot as plt
import numpy as np

print('Upload Bar Data:')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    bar_df = pd.read_csv(filename)

print('Upload Pedestrian Data:')
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    pedestrian_df = pd.read_csv(filename)



pedestrian_copy = pedestrian_df.copy()
bar_copy = bar_df.copy()

Upload Bar Data:


Saving clean_bar_data (1).csv to clean_bar_data (1).csv
Upload Pedestrian Data:


Saving clean_pedestrian_data (1).csv to clean_pedestrian_data (1).csv


In [30]:
pedestrian_df = pedestrian_copy
bar_df = bar_copy

In [31]:
# Join two tables --------------------------------------------------------------------------------
df_joined = pedestrian_df.merge(bar_df, how='cross')
    #A cross join pairs every sensor with every bar, so it gives 100 × 304 = 30,400 rows. Each row is one sensor–bar pair, with both sets of coordinates sitting side by side.


# Determining distance between sensors and bars. --------------------------------------------------------------------------------
    #Haversine: Great-circle distance in metres of the arc between two sets of lat/lon points (works on whole columns).
    #To draw a circle around the entire sphere that intercepts the two points and then measure the difference between the them.

def haversine_meters(lat1, lon1, lat2, lon2):

    R = 6_371_000                                                               # Earth's radius in metres
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2   #Equation
    return 2 * R * np.arcsin(np.sqrt(a))

df_joined['distance_meters'] = haversine_meters(df_joined['Latitude'], df_joined['Longitude'],
                                  df_joined['bar_latitude'], df_joined['bar_longitude'])
    #Comparing the sensors location to the bars location to determine the distance between them per row

'''df_joined['distance_meters'].describe()   '''
    #Checking distance to ensure that min and max are appropriate.
    #Min (7m) is the minimum distance between any sensor and any bar, Max (7km) is inside an acceptable range for the datas scope




"df_joined['distance_meters'].describe()   "

In [32]:
#Total seats by sensor -------------------------------------------------------------------------------
#Function takes a list of radius and adds a columns to DF of number of seats within those radius of each sensor.

def add_seats(list_r, pedestrian_df, df_joined):

  pedestrian_seats = pedestrian_df.copy()   #Copy dataframe and set to variable outside iteration

  for r in list_r:                          #Iterate through list
      sensors_meter_r = df_joined[df_joined['distance_meters'] <= r]
          #Filters to keep only sensor bar pairs within radius.

      col_name_1 = (f"seats_{r}r_inc_outliers")
      col_name_2 = (f"seats_{r}r_excl_outliers")
          #Define Columns names as variables per iteration

      sps = sensors_meter_r.groupby('location_id').agg(**{col_name_1: ('Number of patrons', 'sum')})
      sps_no_outlier = sensors_meter_r[~ sensors_meter_r['outlier_venue']].groupby('location_id').agg(**{col_name_2: ('Number of patrons', 'sum')})
          #Total number of seats of each bar within a sensors haversine radius.

          #LLM: Dynamic Aggregation Keys: When passing dynamic column names to .agg(),
          #Python doesn't allow variable names as keyword parameters directly (like agg(col_name_1 = ...)).
          #Instead, we can unpack a dictionary using **{col_name_1: (...)}.


      pedestrian_seats = pedestrian_seats.merge(sps, how='left', on='location_id').fillna({col_name_1: 0})
      pedestrian_seats = pedestrian_seats.merge(sps_no_outlier, how='left', on='location_id').fillna({col_name_2: 0})
          #add each seat total back to sensor data frame.
  return pedestrian_seats

pedestrian_seats = add_seats([200, 300], pedestrian_df, df_joined)
display(pedestrian_seats.describe())

'''
Orginal Code without function:
sensors_200m_r = df_joined[df_joined['distance_meters'] <= 200]
    #Filters to keep only sensor bar pairs within 200m.

sensors_300m_r = df_joined[df_joined['distance_meters'] <= 300]
    #Filter to keep only sensor bar pairs within 300m.

sps200 = sensors_200m_r.groupby('location_id').agg(seats_200r = ('Number of patrons', 'sum'))
sps200_no_outlier = sensors_200m_r[~ sensors_200m_r['outlier_venue']].groupby('location_id').agg(seats_200r_no_outlier = ('Number of patrons', 'sum'))
    #Total number of seats of each bar within a sensors haversine radious.

pedestrian_seats = pedestrian_df.merge(sps200, how='left', on='location_id').fillna({'seats_200r': 0})
pedestrian_seats = pedestrian_seats.merge(sps200_no_outlier, how='left', on='location_id').fillna({'seats_200r_no_outlier': 0})
    #add each seat total back to sensor data frame.
'''


,location_id,early,late,Latitude,Longitude,seats_200r_inc_outliers,seats_200r_excl_outliers,seats_300r_inc_outliers,seats_300r_excl_outliers
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.00000
mean,80.890000,733.057500,355.180208,-37.813006,144.960310,2112.750000,2032.35000,4725.350000,4455.15000
std,57.585228,732.996832,388.056380,0.006824,0.009329,2285.050354,2217.64581,4080.810002,3941.18575
min,1.000000,5.750000,2.520833,-37.825910,144.929734,0.000000,0.00000,0.000000,0.00000
25%,35.750000,204.843750,66.265625,-37.817950,144.955261,305.000000,305.00000,1051.250000,882.50000
50%,66.500000,454.791667,218.718750,-37.813802,144.963023,1383.500000,1254.00000,4015.000000,3871.50000
75%,133.250000,1017.135417,486.804688,-37.809857,144.966112,3152.250000,2956.00000,6954.750000,6755.25000
max,209.000000,3052.083333,1548.562500,-37.794324,144.974677,7586.000000,7432.00000,15877.000000,14777.00000


"\nOrginal Code without function:\nsensors_200m_r = df_joined[df_joined['distance_meters'] <= 200]\n    #Filters to keep only sensor bar pairs within 200m.\n\nsensors_300m_r = df_joined[df_joined['distance_meters'] <= 300]\n    #Filter to keep only sensor bar pairs within 300m.\n\nsps200 = sensors_200m_r.groupby('location_id').agg(seats_200r = ('Number of patrons', 'sum'))\nsps200_no_outlier = sensors_200m_r[~ sensors_200m_r['outlier_venue']].groupby('location_id').agg(seats_200r_no_outlier = ('Number of patrons', 'sum'))\n    #Total number of seats of each bar within a sensors haversine radious.\n\npedestrian_seats = pedestrian_df.merge(sps200, how='left', on='location_id').fillna({'seats_200r': 0})\npedestrian_seats = pedestrian_seats.merge(sps200_no_outlier, how='left', on='location_id').fillna({'seats_200r_no_outlier': 0})\n    #add each seat total back to sensor data frame.\n"

In [43]:
# Find 0 values, sensors with 0 seats for patrons within radius.
    # Check each column condition individually and combine them using '|' (logical OR in pandas)
zero_seats_mask = (
    (pedestrian_seats['seats_200r_inc_outliers'] == 0) |
    (pedestrian_seats['seats_200r_excl_outliers'] == 0) |
    (pedestrian_seats['seats_300r_inc_outliers'] == 0) |
    (pedestrian_seats['seats_300r_excl_outliers'] == 0)
)

    # Filter the DataFrame using the mask directly and print the counts
filtered_seats = pedestrian_seats[zero_seats_mask]
print("Number of rows matching the condition:", len(filtered_seats))
print((filtered_seats[['seats_200r_inc_outliers', 'seats_200r_excl_outliers',
                       'seats_300r_inc_outliers', 'seats_300r_excl_outliers']] == 0).sum())
    # 14 0 values for 200m r and 5 for 300m r. Indicating the sensors without any seats surrounding them.

#Find sensors that fall outside of the bar dataframes geograpgic min and max area.
def check_range(sensors):
    #Function checks a data frame against min and max bar long/lan, returns results outside
    results = []

    lat_min = bar_df['bar_latitude'].min()
    lat_max = bar_df['bar_latitude'].max()
    long_min = bar_df['bar_longitude'].min()
    long_max = bar_df['bar_longitude'].max()

    for i, row in sensors.iterrows():
        #a method used to iterate over the rows of a DataFrame as (index, Series) pairs

        lat_inside = lat_min <= row['Latitude'] <= lat_max
        lon_inside = long_min <= row['Longitude'] <= long_max   #create booleans for if they are inside or outside of range

        results.append({                                         #Append results to a dedicated list of dicts.
            'location_id': row['location_id'],
            'lat_inside_range': lat_inside,
            'lon_inside_range': lon_inside,
            'fully_inside_range': lat_inside and lon_inside
        })

    results = pd.DataFrame(results)                         #Convert list of dict into a DF

    results = results[~ results['fully_inside_range']]       # Only show results outside of datafram

    return(results)


outside_sensor = check_range(pedestrian_seats)

pedestrian_seats['outside_bar_range'] = pedestrian_seats['location_id'].isin(outside_sensor['location_id'])
    #Flag the sensors that lay outside of the bars max\min long\lat.

pedestrian_seats = pedestrian_seats[~ pedestrian_seats['outside_bar_range']]
pedestrian_seats= pedestrian_seats.drop(columns = 'outside_bar_range')
    #Drop irrelevant rows and loss outside range column

'''
2 Sensors came back outside range, both with 0 seat values for every assigned radius.
Since the outside sensors don't have any seats assigned to them, they would be identified as optimal places
for a venue as there is an uncontested amount of foot traffic. This of course is only the result of the limited database.
The sensors fall outside of the area being investigated and have been dropped from data accordingly.
City of Melbourne isn't a perfect rectangle, noting a limitation of the data and this method.
'''


Number of rows matching the condition: 12
seats_200r_inc_outliers     12
seats_200r_excl_outliers    12
seats_300r_inc_outliers      3
seats_300r_excl_outliers     3
dtype: int64


/tmp/ipykernel_2923/3326827096.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  pedestrian_seats['outside_bar_range'] = pedestrian_seats['location_id'].isin(outside_sensor['location_id'])


"    \n2 Sensors came back outside range, both with 0 seat values for every assigned radius.\nSince the outside sensors don't have any seats assigned to them, they would be identified as optimal places\nfor a venue as there is an uncontested amount of foot traffic. Sensors have been dropped from data accordingly.\nCity of Melbourne isn't a perfect rectangle, noting a limitation of the data and this method.\n"